# Credit Risk Prediction Project


## Step 1: Loading Data into SQLite

The raw dataset (`credit_risk_training.csv`, 150,000 records) is loaded into a SQLite database using pandas. The data is stored in a table named `borrowers`, which serves as the foundation for all subsequent SQL based exploration and cleaning.


In [ ]:
import pandas as pd
import sqlite3

df = pd.read_csv("credit_risk_training.csv")
conn = sqlite3.connect("credit_risk.db")
df.to_sql("borrowers", conn, if_exists="replace", index=False)
conn.close()


In [ ]:
conn = sqlite3.connect("credit_risk.db")
query = "SELECT COUNT(*) AS total_rows FROM borrowers"
print(pd.read_sql(query, conn))


## Step 2: Data Quality Checks and Exploratory Analysis

Before any modeling, the dataset is checked for missing values and anomalies using SQL queries:

- **Missing values**: `MonthlyIncome` has approximately 29,731 missing entries (about 20% of records), and `NumberOfDependents` has approximately 3,924 missing entries (about 2.6%).
- **Anomalies**: A small number of records have `age` recorded as 0, treated as a data entry error. The late payment columns contain values of 96 and 98, which appear to be placeholder or error codes rather than genuine counts.
- **Cleaning step**: A new table, `borrowers_clean`, is created where age 0 values are replaced with the average age, the 96/98 codes are converted to NULL, and missing dependent counts are filled with 0.


In [ ]:
query_missing = """
SELECT
  SUM(CASE WHEN MonthlyIncome IS NULL THEN 1 ELSE 0 END) AS missing_income,
  SUM(CASE WHEN NumberOfDependents IS NULL THEN 1 ELSE 0 END) AS missing_dependents
FROM borrowers
"""
print(pd.read_sql(query_missing, conn))


In [ ]:
query_anomaly = """
SELECT
  SUM(CASE WHEN age = 0 THEN 1 ELSE 0 END) AS age_zero,
  SUM(CASE WHEN "NumberOfTime30-59DaysPastDueNotWorse" >= 96 THEN 1 ELSE 0 END) AS weird_30_59,
  SUM(CASE WHEN NumberOfTimes90DaysLate >= 96 THEN 1 ELSE 0 END) AS weird_90,
  SUM(CASE WHEN "NumberOfTime60-89DaysPastDueNotWorse" >= 96 THEN 1 ELSE 0 END) AS weird_60_89
FROM borrowers
"""
print(pd.read_sql(query_anomaly, conn))


In [ ]:
create_clean = """
CREATE TABLE borrowers_clean AS
SELECT
  CustomerID,
  SeriousDlqin2yrs,
  RevolvingUtilizationOfUnsecuredLines,
  CASE WHEN age = 0 THEN (SELECT AVG(age) FROM borrowers WHERE age > 0) ELSE age END AS age,
  CASE WHEN "NumberOfTime30-59DaysPastDueNotWorse" >= 96 THEN NULL ELSE "NumberOfTime30-59DaysPastDueNotWorse" END AS late_30_59,
  DebtRatio,
  MonthlyIncome,
  NumberOfOpenCreditLinesAndLoans,
  CASE WHEN NumberOfTimes90DaysLate >= 96 THEN NULL ELSE NumberOfTimes90DaysLate END AS late_90_plus,
  NumberRealEstateLoansOrLines,
  CASE WHEN "NumberOfTime60-89DaysPastDueNotWorse" >= 96 THEN NULL ELSE "NumberOfTime60-89DaysPastDueNotWorse" END AS late_60_89,
  COALESCE(NumberOfDependents, 0) AS NumberOfDependents
FROM borrowers
"""
conn.execute(create_clean)
conn.commit()


In [ ]:
query_age = """
SELECT
  CASE
    WHEN age < 30 THEN 'under_30'
    WHEN age < 45 THEN '30_to_45'
    WHEN age < 60 THEN '45_to_60'
    ELSE 'over_60'
  END AS age_group,
  COUNT(*) AS total,
  ROUND(AVG(SeriousDlqin2yrs) * 100, 2) AS default_rate_percent
FROM borrowers_clean
GROUP BY age_group
ORDER BY default_rate_percent DESC
"""
print(pd.read_sql(query_age, conn))


In [ ]:
query_income = """
SELECT
  CASE
    WHEN MonthlyIncome IS NULL THEN 'unknown'
    WHEN MonthlyIncome < 3000 THEN 'under_3000'
    WHEN MonthlyIncome < 6000 THEN '3000_to_6000'
    WHEN MonthlyIncome < 10000 THEN '6000_to_10000'
    ELSE 'over_10000'
  END AS income_group,
  COUNT(*) AS total,
  ROUND(AVG(SeriousDlqin2yrs) * 100, 2) AS default_rate_percent
FROM borrowers_clean
GROUP BY income_group
ORDER BY default_rate_percent DESC
"""
print(pd.read_sql(query_income, conn))


## Step 3: Feature Engineering and Model Training

Remaining missing values in `MonthlyIncome` and the late payment columns are imputed using the median, which is more robust to outliers than the mean.

Three additional features are engineered to capture risk signals not directly present in the raw columns:
- `total_past_due`: the sum of all late payment counts across the three time windows.
- `has_severe_delinquency`: a binary flag indicating whether the borrower has ever been 90+ days late.
- `income_per_dependent`: monthly income divided by household size, capturing financial strain per dependent.

The dataset is split into training (75%) and test (25%) sets using stratified sampling to preserve the roughly 7% default rate in both splits.

Two models are trained and compared: a Logistic Regression baseline with balanced class weights, and a Random Forest, also with balanced class weights, which achieves a noticeably higher AUC.


In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

conn = sqlite3.connect("credit_risk.db")
df = pd.read_sql("SELECT * FROM borrowers_clean", conn)
conn.close()

print(df.shape)
print(df.isna().sum())


In [ ]:
df["MonthlyIncome"] = df["MonthlyIncome"].fillna(df["MonthlyIncome"].median())
df["late_30_59"] = df["late_30_59"].fillna(df["late_30_59"].median())
df["late_90_plus"] = df["late_90_plus"].fillna(df["late_90_plus"].median())
df["late_60_89"] = df["late_60_89"].fillna(df["late_60_89"].median())

print(df.isna().sum())


In [ ]:
df["total_past_due"] = df["late_30_59"] + df["late_60_89"] + df["late_90_plus"]
df["has_severe_delinquency"] = (df["late_90_plus"] > 0).astype(int)
df["income_per_dependent"] = df["MonthlyIncome"] / (df["NumberOfDependents"] + 1)

print(df[["total_past_due", "has_severe_delinquency", "income_per_dependent"]].describe())


In [ ]:
feature_cols = [
    "RevolvingUtilizationOfUnsecuredLines", "age", "late_30_59", "DebtRatio",
    "MonthlyIncome", "NumberOfOpenCreditLinesAndLoans", "late_90_plus",
    "NumberRealEstateLoansOrLines", "late_60_89", "NumberOfDependents",
    "total_past_due", "has_severe_delinquency", "income_per_dependent"
]

X = df[feature_cols]
y = df["SeriousDlqin2yrs"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
log_reg.fit(X_train_scaled, y_train)

log_reg_probs = log_reg.predict_proba(X_test_scaled)[:, 1]
log_reg_auc = roc_auc_score(y_test, log_reg_probs)

print("Logistic Regression AUC:", round(log_reg_auc, 4))


In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, class_weight="balanced",
    random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

rf_probs = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_probs)

print("Random Forest AUC:", round(rf_auc, 4))


In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances)


In [ ]:
rf_preds = (rf_probs >= 0.5).astype(int)
print(classification_report(y_test, rf_preds))
print(confusion_matrix(y_test, rf_preds))


In [ ]:
results = X_test.copy()
results["actual_default"] = y_test.values
results["predicted_probability"] = rf_probs
results["predicted_default"] = rf_preds

results.to_csv("model_predictions.csv", index=False)
print(results.head())


## Step 4: Business Decision Analysis

A threshold sensitivity analysis is performed in Python. For threshold values ranging from 0.05 to 0.95, the resulting false positives (good customers incorrectly flagged as risky) and false negatives (defaults the model fails to catch) are calculated, along with an estimated financial loss from missed defaults, assuming an average loan size of 1,000 currency units per customer. This produces a clear trade off curve between catching more defaults and rejecting fewer good customers, which is the basis of the second Power BI dashboard page.


In [ ]:
thresholds = np.arange(0.05, 1.0, 0.05)
avg_loan_amount = 1000  # business assumption: average loan size per customer

rows = []
for t in thresholds:
    preds_t = (rf_probs >= t).astype(int)
    tp = ((preds_t == 1) & (y_test == 1)).sum()
    fp = ((preds_t == 1) & (y_test == 0)).sum()
    fn = ((preds_t == 0) & (y_test == 1)).sum()
    tn = ((preds_t == 0) & (y_test == 0)).sum()
    rows.append({
        "threshold": round(t, 2),
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn,
        "estimated_loss_missed_defaults": fn * avg_loan_amount,
        "good_customers_rejected": fp
    })

threshold_df = pd.DataFrame(rows)
threshold_df.to_csv("threshold_analysis.csv", index=False)
print(threshold_df)
